# Qualidade de `onboarding.csv`

Diagnóstico do questionário de entrada pelos mesmos cinco critérios usados nos outros arquivos, mais a checagem de privacidade do texto livre:

| Critério | Pergunta |
|---|---|
| **Completude** | Está tudo lá? Respostas vazias ou sem conteúdo, e quem deveria estar no arquivo |
| **Validade** | Categorias, datas e números no domínio esperado? Texto livre que na prática é categoria converte? |
| **Consistência** | O texto concorda com as categorias e com os outros arquivos? |
| **Privacidade** | Há e-mail, telefone, CPF ou @ de terceiros no texto? |
| **Unicidade** | Cada conta responde uma vez só? |
| **Temporalidade** | As datas estão no período certo, antes do corte, no ritmo da base? |

**Só diagnóstico:** nada é corrigido aqui. Cada achado vai para uma tabela-resumo em `saidas/`, com status
(`ok`, `esperado` ou `atenção`) e tratamento sugerido. Nada em `dados/brutos/` é alterado (hash conferido no início e no fim).
Transformar o texto em variáveis (extração com IA generativa, objetivo 5 do case) é outra etapa e não é feito aqui.

**Decisões já registradas que valem aqui** (`decisoes.md`): contas de teste e cópias de `usuarios.csv` removidas; datas
dos logs em horário de Brasília; nenhuma informação posterior ao corte.

**Privacidade:** texto livre aparece só como contagem e respostas mais frequentes, nunca ao lado do `user_id`.

In [1]:
ARQUIVO = "onboarding.csv"

import hashlib
import re
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 70)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BRUTOS = ROOT / "dados" / "brutos"
PROCESSADOS = ROOT / "dados" / "processados"
SAIDAS = ROOT / "saidas"
CORTE = pd.Timestamp("2026-09-30")
FIM_CORTE = CORTE + pd.Timedelta(days=1)

ARQUIVOS = [ARQUIVO, "usuarios.csv"]

def sha1(path):
    return hashlib.sha1(path.read_bytes()).hexdigest()

hash_inicio = {f: sha1(BRUTOS / f) for f in ARQUIVOS}

achados = []

def achado(criterio, checagem, resultado, linhas, tratamento, status=None):
    # status: ok (nada encontrado), esperado (encontrado, mas é comportamento do negócio) ou atenção.
    if status is None:
        status = "ok" if linhas == 0 else "atenção"
    achados.append({"criterio": criterio, "checagem": checagem, "status": status,
                    "resultado": resultado, "linhas_afetadas": int(linhas), "tratamento_sugerido": tratamento})

# respostas sem conteúdo (vazio disfarçado), pelo checklist da skill
SEM_CONTEUDO = {"nada", "ok", "x", ".", "-", "asdf", "kkkk", "sla", "nao sei", "não sei", "nan", "null", "none", "n/a"}

def sem_conteudo(s):
    t = s.str.strip().str.lower()
    so_simbolo = t.ne("") & ~t.str.contains(r"[a-z0-9à-ú]", regex=True)   # só emoji ou pontuação
    return t.isin(SEM_CONTEUDO) | so_simbolo

PII = {
    "e-mail": r"[\w.+-]+@[\w-]+\.[a-z]{2,}",
    "telefone": r"\(?\b\d{2}\)?\s?9?\d{4}-?\d{4}\b",
    "CPF": r"\b\d{3}\.?\d{3}\.?\d{3}-?\d{2}\b",
    "@ de perfil": r"(?<![\w.])@\w{3,}",
}

def contar_pii(s):
    return {nome: int(s.str.contains(rx, regex=True, case=False).sum()) for nome, rx in PII.items()}

brutos_usuarios = pd.read_csv(BRUTOS / "usuarios.csv", dtype=str, keep_default_na=False, usecols=["user_id", "email"])
email = brutos_usuarios.set_index("user_id")["email"].str.strip()
TESTE = set(email.index[email.str.lower().str.endswith("@vitaliza.com.br")])
COPIA = set(email.index[email != email.str.lower()])
usuarios = pd.read_parquet(PROCESSADOS / "usuarios_limpa.parquet", columns=["user_id", "data_cadastro", "canal_aquisicao"]).set_index("user_id")

def contas_invalidas(df, criterio="Completude"):
    n_teste, n_copia = int(df["user_id"].isin(TESTE).sum()), int(df["user_id"].isin(COPIA).sum())
    sem_cad = int((~df["user_id"].isin(email.index)).sum())
    achado(criterio, "linhas de contas de teste, cópias ou sem cadastro",
           f"teste: {n_teste}; cópias: {n_copia}; sem cadastro: {sem_cad}", n_teste + n_copia + sem_cad, "Se houver: excluir")

## 0. Leitura

In [2]:
onb = pd.read_csv(BRUTOS / ARQUIVO, dtype=str, keep_default_na=False)
assert len(onb) == 63_929  # cabeçalho da seção 10 do dicionário
N = len(onb)
print(f"onboarding: {N:,} linhas x {onb.shape[1]} colunas -> {list(onb.columns)}")
onb["data"] = pd.to_datetime(onb["respondido_em"], format="%Y-%m-%d", errors="coerce")
onb["ano"] = onb["data"].dt.year
onb["canal_aquisicao"] = onb["user_id"].map(usuarios["canal_aquisicao"])
onb[["nivel", "dias_por_semana"]].value_counts().head(5)

onboarding: 63,929 linhas x 7 colunas -> ['onboarding_id', 'user_id', 'respondido_em', 'objetivo_texto', 'como_conheceu', 'nivel', 'dias_por_semana']


nivel          dias_por_semana
intermediario  3                  4114
iniciante      3                  3919
               2                  3069
intermediario  2                  3063
               4                  3041
Name: count, dtype: int64

## 1. Completude

O objetivo e o como conheceu são opcionais (dicionário); o nível e os dias por semana, o dicionário não diz. Além do vazio,
conta como vazio a resposta sem conteúdo (`nada`, `ok`, `x`, `.`, `asdf`, `kkkk`, `sla`, `nao sei`, só emoji).

In [3]:
TEXTO = ["objetivo_texto", "como_conheceu", "nivel", "dias_por_semana"]
falta = pd.DataFrame({
    "vazio": [int((onb[c].str.strip() == "").sum()) for c in TEXTO],
    "sem conteúdo": [int(sem_conteudo(onb[c]).sum()) for c in TEXTO],
}, index=TEXTO)
falta["% sem resposta útil"] = (100 * falta.sum(axis=1) / N).round(1)
display(falta)
print("respostas sem conteúdo mais comuns no objetivo:",
      onb.loc[sem_conteudo(onb["objetivo_texto"]), "objetivo_texto"].value_counts().head(12).to_dict())
por_ano = pd.DataFrame({c: ((onb[c].str.strip() == "") | sem_conteudo(onb[c])).groupby(onb["ano"]).mean().mul(100).round(1)
                        for c in TEXTO})
display(por_ano)

,vazio,sem conteúdo,% sem resposta útil
objetivo_texto,5467,13768,30.1
como_conheceu,4586,11515,25.2
nivel,4513,0,7.1
dias_por_semana,6368,0,10.0


respostas sem conteúdo mais comuns no objetivo: {'nada': 1426, '😂': 1413, '.': 1398, 'x': 1397, 'sla': 1396, 'kkkk': 1365, 'asdf': 1365, 'nao sei': 1361, 'ok': 1325, '-': 1322}


,objetivo_texto,como_conheceu,nivel,dias_por_semana
ano,,,,
2020,28.9,28.0,6.7,10.6
2021,29.9,26.4,7.5,11.1
2022,29.3,24.8,7.5,9.9
2023,30.1,24.3,8.4,10.0
2024,30.4,25.1,6.9,9.7
2025,29.8,25.1,7.1,9.8
2026,30.3,25.2,6.9,10.2


In [4]:
for c in TEXTO:
    vazio, lixo = int(falta.loc[c, "vazio"]), int(falta.loc[c, "sem conteúdo"])
    faixa = f"{por_ano[c].min()}% a {por_ano[c].max()}% por ano"
    if lixo:
        achado("Completude", f"{c}: vazio ou sem conteúdo",
               f"{vazio:,} vazios e {lixo:,} sem conteúdo ({falta.loc[c, '% sem resposta útil']}%); {faixa}", vazio + lixo,
               "Tratar a resposta sem conteúdo como não respondida (somar aos vazios); não imputar")
    else:
        achado("Completude", f"{c}: vazio", f"{vazio:,} ({100 * vazio / N:.1f}%); {faixa}", vazio,
               "Manter como não respondido; não imputar", status="esperado")
contas_invalidas(onb)
fora = len(set(usuarios.index) - set(onb["user_id"])) + len(set(onb["user_id"]) - set(usuarios.index))
achado("Completude", "contas do cadastro limpo sem questionário (e o contrário)", f"{fora}", fora, "Nenhum")

## 2. Validade

`nivel` tem três valores. `dias_por_semana` é "texto livre", mas na prática é um número de 1 a 7 escrito de várias formas.

In [5]:
nivel_fora = int((~onb["nivel"].isin({"iniciante", "intermediario", "avancado", ""})).sum())
achado("Validade", "nivel fora de iniciante, intermediario, avancado", f"{nivel_fora}", nivel_fora, "Nenhum")

dias = onb["dias_por_semana"].str.strip().str.lower()
numero = dias.str.extract(r"^([1-7])(?:\s*x|\s*vezes)?$")[0]
numero = numero.where(dias != "todos", "7")
print("grafias:", dias.value_counts().to_dict())
nao_converte = int((numero.isna() & (dias != "")).sum())
fora_padrao = int((dias.ne("") & ~dias.str.fullmatch(r"[1-7]")).sum())
assert nao_converte == 0
achado("Validade", "dias_por_semana escrito de formas diferentes",
       f"{dias[dias != ''].nunique()} grafias para 7 valores (3, 3x, 3 vezes, todos...); {fora_padrao:,} fora do número puro; "
       f"{nao_converte} sem conversão", fora_padrao,
       "Converter para número de 1 a 7: 'N', 'Nx' e 'N vezes' -> N; 'todos' -> 7; vazio -> ausente")
falha_data = int(onb["data"].isna().sum())
achado("Validade", "respondido_em que não converte", f"{falha_data}", falha_data, "Nenhum")

grafias: {'3': 10268, '2': 7576, '4': 7437, '': 6368, '5': 3649, '3 vezes': 3450, '3x': 3443, '1': 2671, '2x': 2558, '4 vezes': 2557, '4x': 2452, '2 vezes': 2443, '5 vezes': 1276, '5x': 1254, '6': 1128, '1x': 925, 'todos': 917, '1 vezes': 906, '7': 798, '6 vezes': 549, '6x': 527, '7x': 395, '7 vezes': 382}


### Como conheceu

Texto livre com poucas respostas muito repetidas, erros de digitação (letras trocadas) e o @ de seis influenciadores. Hipótese:
cada grafia rara é um erro de digitação de uma resposta frequente. Teste: a resposta frequente mais parecida (difflib, corte
0,8) tem de ser do mesmo canal de aquisição da conta.

In [6]:
import difflib

cc = onb["como_conheceu"].str.strip()
# espaço trocado de lugar em volta do @ ("da@ carolfit" -> "da @carolfit")
cc = cc.str.replace(r"@\s+(\w)", r"@\1", regex=True).str.replace(r"(\w)@(\w)", r"\1 @\2", regex=True)
handle = cc.str.extract(r"@(\w+)")[0]
contagem_handle = handle.value_counts()
HANDLES = list(contagem_handle[contagem_handle >= 500].index)
print(f"@ citados: {handle.nunique()} grafias; frequentes (>= 500): {len(HANDLES)}")
def handle_certo(h):
    if pd.isna(h):
        return h
    perto = difflib.get_close_matches(h, HANDLES, n=1, cutoff=0.8)
    return perto[0] if perto else None
handle_norm = handle.map({h: handle_certo(h) for h in handle.dropna().unique()})
sem_handle = int((handle.notna() & handle_norm.isna()).sum())
print("grafias de @ sem correspondência:", sem_handle)

import unicodedata

def sem_acento(t):
    return "".join(ch for ch in unicodedata.normalize("NFD", t) if unicodedata.category(ch) != "Mn")

forma = cc.str.lower().str.replace(r"@\w+", "@X", regex=True)
contagem = forma[forma.ne("") & ~sem_conteudo(cc)].value_counts()
# respostas frequentes; variantes só de acento ("ja conhecia", "já conhecia") ficam com a grafia mais comum
frequentes = contagem[contagem >= 100]
POR_CHAVE = {}
for t, n in frequentes.items():
    POR_CHAVE.setdefault(sem_acento(t), t)
CANONICAS = sorted(set(POR_CHAVE.values()))
CHAVES = list(POR_CHAVE)
def canonica(t):
    k = sem_acento(t)
    if k in POR_CHAVE:
        return POR_CHAVE[k]
    perto = difflib.get_close_matches(k, CHAVES, n=1, cutoff=0.8)
    return POR_CHAVE[perto[0]] if perto else None
mapa = {t: canonica(t) for t in contagem.index}
onb["como_conheceu_norm"] = forma.map(mapa)
sem_mapa = sorted(t for t, c in mapa.items() if c is None)
print(f"{len(contagem)} grafias com conteúdo -> {len(CANONICAS)} respostas; sem correspondência: {len(sem_mapa)} "
      f"({int(contagem[sem_mapa].sum()) if sem_mapa else 0} respostas): {sem_mapa[:10]}")

canal_da_resposta = onb.dropna(subset=["como_conheceu_norm"]).groupby("como_conheceu_norm")["canal_aquisicao"].agg(
    lambda s: s.value_counts().index[0])
mesmo_canal = onb["como_conheceu_norm"].map(canal_da_resposta) == onb["canal_aquisicao"]
erro_digitacao = onb["como_conheceu_norm"].notna() & (forma != onb["como_conheceu_norm"])
print(f"respostas corrigidas por erro de digitação: {erro_digitacao.sum():,}; delas no mesmo canal da conta: "
      f"{(mesmo_canal & erro_digitacao).sum():,}")
display(onb.groupby("como_conheceu_norm")["canal_aquisicao"].agg(lambda s: s.value_counts().to_dict()).to_frame("canal"))
achado("Validade", "como_conheceu com erro de digitação",
       f"{len(contagem)} grafias para {len(CANONICAS)} respostas; {erro_digitacao.sum():,} com letras trocadas, todas "
       f"atribuídas a uma resposta do mesmo canal de aquisição ({(mesmo_canal & erro_digitacao).mean() / max(erro_digitacao.mean(), 1e-9):.0%}); "
       f"{len(HANDLES)} influenciadores em {handle.nunique()} grafias de @", int(erro_digitacao.sum()),
       "Padronizar pela resposta frequente mais parecida (tabela deste notebook) e o @ pelo do influenciador mais parecido")

@ citados: 70 grafias; frequentes (>= 500): 6
grafias de @ sem correspondência: 0
300 grafias com conteúdo -> 20 respostas; sem correspondência: 0 (0 respostas): []
respostas corrigidas por erro de digitação: 4,088; delas no mesmo canal da conta: 4,088


,canal
como_conheceu_norm,
a @X indicou nos stories,{'influenciador': 2584}
a empresa deu desconto,{'parceria_corporativa': 1340}
anúncio no instagram,{'meta_ads': 2892}
anúncio no youtube,{'google_ads': 2463}
benefício do trabalho,{'parceria_corporativa': 1313}
convênio da empresa,{'parceria_corporativa': 1342}
cupom da @X,{'influenciador': 2634}
indicação de amigo,{'indicacao': 1496}
já conhecia,{'organico': 3360}


## 3. Consistência

Como conheceu (declarado pela pessoa) contra o canal atribuído pelo marketing; data da resposta contra a do cadastro.

In [7]:
respondido = onb["como_conheceu_norm"].notna()
diverge = int((respondido & ~mesmo_canal).sum())
achado("Consistência", "como conheceu (declarado) diferente do canal de aquisição (marketing)",
       f"{diverge:,} de {respondido.sum():,} respostas; cada resposta pertence a um único canal", diverge,
       "Nenhum: o como conheceu repete o canal_aquisicao e não traz informação nova")
dif = (onb["data"] - onb["user_id"].map(usuarios["data_cadastro"])).dt.days
print("dias entre resposta e cadastro:", dif.value_counts().to_dict())
achado("Consistência", "resposta em dia diferente do cadastro", f"{int((dif != 0).sum())}", int((dif != 0).sum()), "Nenhum")

dias entre resposta e cadastro: {0: 63929}


## 4. Privacidade

In [8]:
pii = {c: contar_pii(onb[c]) for c in ["objetivo_texto", "como_conheceu"]}
display(pd.DataFrame(pii))
pessoal = sum(v for c in pii.values() for k, v in c.items() if k != "@ de perfil")
achado("Privacidade", "e-mail, telefone ou CPF no texto", f"{pessoal}", pessoal, "Nenhum")
n_handle = pii["como_conheceu"]["@ de perfil"]
achado("Privacidade", "@ de terceiros no como conheceu",
       f"{n_handle:,} respostas citam o @ de {len(HANDLES)} influenciadores parceiros (perfis públicos de divulgação)", n_handle,
       "Manter o nome do influenciador como categoria; não é dado pessoal de cliente", status="esperado")

,objetivo_texto,como_conheceu
e-mail,0,0
telefone,0,0
CPF,0,0
@ de perfil,0,7729


## 5. Unicidade

In [9]:
dup = {c: int(onb[c].duplicated().sum()) for c in ["onboarding_id", "user_id"]}
linha = int(onb.drop(columns=["onboarding_id"]).duplicated().sum())
achado("Unicidade", "onboarding_id ou user_id repetido", f"{dup}", sum(dup.values()), "Nenhum")
achado("Unicidade", "linha repetida sem o onboarding_id", f"{linha}", linha, "Nenhum")

## 6. Temporalidade

As respostas acompanham os cadastros (mesmo dia). O questionário não tem coluna que mude depois da resposta.

In [10]:
print(f"respondido_em de {onb['data'].min().date()} a {onb['data'].max().date()}")
fora_periodo = int(((onb["data"] < pd.Timestamp("2020-09-01")) | (onb["data"] > CORTE)).sum())
achado("Temporalidade", "resposta antes do lançamento ou depois do corte", f"{fora_periodo}", fora_periodo, "Nenhum")

respondido_em de 2020-09-01 a 2026-09-30


## Resumo

In [11]:
resumo = pd.DataFrame(achados)
resumo["% do arquivo"] = (100 * resumo["linhas_afetadas"] / N).round(2)
resumo.to_csv(SAIDAS / f"qualidade_{Path(ARQUIVO).stem}.csv", index=False)
print(resumo["status"].value_counts().to_dict())
resumo

{'ok': 10, 'atenção': 4, 'esperado': 3}


,criterio,checagem,status,resultado,linhas_afetadas,tratamento_sugerido,% do arquivo
0,Completude,objetivo_texto: vazio ou sem conteúdo,atenção,"5,467 vazios e 13,768 sem conteúdo (30.1%); 28.9% a 30.4% por ano",19235,Tratar a resposta sem conteúdo como não respondida (somar aos vazi...,30.09
1,Completude,como_conheceu: vazio ou sem conteúdo,atenção,"4,586 vazios e 11,515 sem conteúdo (25.2%); 24.3% a 28.0% por ano",16101,Tratar a resposta sem conteúdo como não respondida (somar aos vazi...,25.19
2,Completude,nivel: vazio,esperado,"4,513 (7.1%); 6.7% a 8.4% por ano",4513,Manter como não respondido; não imputar,7.06
3,Completude,dias_por_semana: vazio,esperado,"6,368 (10.0%); 9.7% a 11.1% por ano",6368,Manter como não respondido; não imputar,9.96
4,Completude,"linhas de contas de teste, cópias ou sem cadastro",ok,teste: 0; cópias: 0; sem cadastro: 0,0,Se houver: excluir,0.00
5,Completude,contas do cadastro limpo sem questionário (e o contrário),ok,0,0,Nenhum,0.00
6,Validade,"nivel fora de iniciante, intermediario, avancado",ok,0,0,Nenhum,0.00
7,Validade,dias_por_semana escrito de formas diferentes,atenção,"22 grafias para 7 valores (3, 3x, 3 vezes, todos...); 24,034 fora ...",24034,"Converter para número de 1 a 7: 'N', 'Nx' e 'N vezes' -> N; 'todos...",37.59
8,Validade,respondido_em que não converte,ok,0,0,Nenhum,0.00
9,Validade,como_conheceu com erro de digitação,atenção,"300 grafias para 20 respostas; 4,088 com letras trocadas, todas at...",4088,Padronizar pela resposta frequente mais parecida (tabela deste not...,6.39


In [12]:
hash_fim = {f: sha1(BRUTOS / f) for f in ARQUIVOS}
assert hash_fim == hash_inicio
print("dados/brutos/ inalterado:", hash_fim == hash_inicio)

dados/brutos/ inalterado: True
